# Day 2: measurements, uncertainty and decisions

Work through this notebook alongside [E1-E7](../../docs/kalman_filter_practical.md), using the same exercise times. It replaces repeated command-line steps, not the experiments. Before changing a value, write a prediction; afterwards, compare it with the plots.

Use **Crazyflie lectures (.venv)** as the kernel. The optional capture cell connects to the drone for motor-free logging; all other cells work on files or simulation. Follow the [motor-free handling and recording checks](../../docs/sensor_recording_contract.md#live-e6-session-and-replay-fallback). Keep a checked shared recording available; after five minutes of unexpected live-data problems, use it. The default simulated data let this notebook run without hardware.

For a new full session, choose a fresh `run_id` and rerun setup. Within a session, edit and rerun an experiment cell directly: each execution chooses a new numbered attempt folder and keeps the selected recording. Earlier results, including partial failed attempts, are preserved. Save this edited notebook with your results, especially the E4/E5 decision cells.


In [ ]:
from pathlib import Path
import csv
import json
import os
import sys
from dataclasses import replace

# Work from the project root, code/, or this notebook's own folder.
root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day2/experiments.py").is_file())
sys.path.insert(0, str(root / "code"))

import numpy as np
from IPython.display import Image, display
from kalman.filter import FilterConfig, PositionVelocityKalmanFilter
from race_common.dynamics import position_velocity_model
from day2.experiments import ExperimentCase, export_prepared
from day2.notebook_tools import capture, select_data, compare, next_attempt

run_id = "trial-01"  # Change before repeating the notebook.
output = root / "day2/notebook" / run_id
output.mkdir(parents=True, exist_ok=False)
seed = 0
print("Python:", sys.executable)
print("Results:", output)


## E1: inspect one prediction and update

The state is `[x, y, z, vx, vy, vz]`. Acceleration drives the prediction; position corrects it. Predict which entries change during each step. Try a different `dt` or position observation, then explain what happens to the estimate and its uncertainty. The [worksheet](../../docs/kalman_filter_practical.md#e1-identify-the-model-and-reproduce-the-baseline) develops the model.


In [ ]:
dt = 0.02  # seconds
transition, acceleration_input = position_velocity_model(dt)
display(transition, acceleration_input)

config = FilterConfig()
estimator = PositionVelocityKalmanFilter(config)
estimator.x[2] = 1.0
estimator.predict(dt, np.array([0.2, 0.0, 0.0]))
print("Predicted state:", estimator.x)
estimator.update_position(np.array([0.02, -0.01, 1.0]))
print("Updated state:", estimator.x)
print("Innovation:", estimator.last_innovation)
print("State standard deviations:", np.sqrt(np.diag(estimator.p)))


## E2: check calibration before tuning

Follow the [45-minute calibration exercise](../../docs/sensor_bias_calibration_exercise.md#e2-core-45-minutes): inspect the available channels, make two separate 60-second stationary recordings, check the acceleration magnitude, and assess one gyro correction on the second recording.

The optional cell below helps analyse the gyro columns. Set the two paths and column names from the actual cfclient CSV headers; values are in degrees/s. Set `stationary_checks_passed` only after the motor, rest and acceleration checks in the guide. Otherwise this cell skips calibration. A prepared position/acceleration recording does not supply raw gyro data.

Compare the held-out mean before and after subtracting the calibration mean. Decide whether a correction is justified, considering the scatter and existing onboard compensation. Record that decision; the cell does not change the drone or its settings.


In [ ]:
calibration_csv = None  # Example: root / "data/imu-calibration.csv"
validation_csv = None   # A separate stationary recording.
gyro_columns = ("gyro.x", "gyro.y", "gyro.z")
stationary_checks_passed = False

def read_gyro(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        rows = csv.DictReader(stream)
        values = np.array([[float(row[column]) for column in gyro_columns] for row in rows])
    if values.ndim != 2 or values.shape[0] < 2 or not np.all(np.isfinite(values)):
        raise ValueError("Need at least two finite three-axis gyro samples.")
    return np.deg2rad(values)

gyro_bias = None  # Clear any earlier result before validating new inputs.
if stationary_checks_passed and calibration_csv is not None and validation_csv is not None:
    if Path(calibration_csv).samefile(validation_csv):
        raise ValueError("Calibration and held-out validation need two separate recording files.")
    training = read_gyro(calibration_csv)
    validation = read_gyro(validation_csv)
    gyro_bias = training.mean(axis=0)
    print("Calibration mean [rad/s]:", gyro_bias)
    print("Calibration sample std [rad/s]:", training.std(axis=0, ddof=1))
    print("Held-out mean before correction [rad/s]:", validation.mean(axis=0))
    print("Held-out mean after correction [rad/s]:", (validation - gyro_bias).mean(axis=0))
else:
    print("Calibration not performed here. Complete E2 with checked live or supplied IMU recordings.")


## Optional: capture one live recording for E3-E6

This replaces the terminal capture command within the existing practical time. Agree who handles the drone and who operates the laptop. Keep hands and loose objects clear of propellers. Prepare the drone as follows:

1. Put the drone on a stable surface. Close cfclient and any other drone-control scripts. Keep hands and loose objects clear of the propellers; do not pick it up yet.
2. Enter your team's assigned URI below. Set `ready_for_motor_stop = True` to confirm step 1, then set `run_live_capture = True` and run the cell.
3. The helper connects, sets `stabilizer.stop=1`, and checks the parameter readback, the supervisor's locked state and all four zero motor outputs. Wait for **STOP VERIFIED** in the cell output before picking up the drone. If verification fails, leave it on the surface and use a checked recording or simulation.
4. Once **STOP VERIFIED** appears, record rest, slow translations and gentle rotations, finishing with another stationary hold. Keep clear of the fitted propellers. The logger continues checking motor outputs during acquisition.

The firmware stop locks the drone until reboot. After motor-free work, place it down, power-cycle it and repeat the normal preflight checks before flight. Setting stop back to `0` does not unlock it. See the [recording procedure and firmware references](../../docs/sensor_recording_contract.md#prepare-the-drone-for-hand-held-recording).

Sampling starts when usable telemetry arrives after stop verification. Use secure handling that prevents propeller contact. Capture lasts 60 seconds after the first usable sample, with a 90-second overall limit including connection and saving. Progress appears in the cell while it runs; the recorded comparison follows afterwards. Interrupting the kernel stops the child logger. The only parameter written is `stabilizer.stop=1`; it is never cleared, including on errors or interruption.

Each capture needs a new `capture_name`; previous files are preserved. Failed, timed-out or interrupted captures retain any partial files but are not selected automatically. To continue offline, set `run_live_capture = False`, rerun this cell and choose a checked recording below, or leave both paths unset for simulation. Do not debug unexpected live-data behaviour for more than five minutes.

The capture supplies raw acceleration, onboard attitude and onboard position, not raw gyro for E2. Position and attitude are already onboard estimates with shared IMU information; agreement with them is not independent accuracy.


In [ ]:
run_live_capture = False  # Opt in after the handling checks above.
ready_for_motor_stop = False  # True after placing the drone down and closing radio clients.
uri = os.environ.get("CF_URI", "")  # Or enter the team's assigned URI here.
capture_name = "capture-01"  # Use a fresh name for another recording.
duration_s = 60  # Allowed range: 1-120 seconds.

captured_recording, captured_metadata = None, None
if run_live_capture:
    captured_recording, captured_metadata = capture(
        output / capture_name, uri, duration_s=duration_s,
        ready_for_motor_stop=ready_for_motor_stop)
    details = json.loads(captured_metadata.read_text(encoding="utf-8"))
    print("Saved samples:", details["prepared_samples"], "status:", details["status"])
    display(Image(filename=str(captured_recording.parent / "live-comparison.png")))
else:
    print("Live capture skipped. Select a saved recording below, or keep the simulation fallback.")


## Choose data once for E3-E6

A successful capture above supplies both paths automatically. To use an existing live or checked shared recording, replace **both** paths below with its `prepared.csv` and `recording.json`. Leave both as `None` for the explicit simulation fallback. If selection fails, the previous dataset is cleared. Correct the paths or choose the simulation fallback, then rerun this cell before continuing.

All configurations in a comparison receive the same observations. Onboard position is already filtered and shares sensor information with onboard attitude. Agreement with it is not independent accuracy; recorded-data RMSE and coverage against truth remain unavailable.


In [ ]:
data = None  # Clear the previous selection before trying new input.
recording = captured_recording  # Or: root / "data/team-live/prepared.csv"
metadata = captured_metadata    # Or: root / "data/team-live/recording.json"

data = select_data(recording, metadata, seed=seed)
print("Input:", "SIMULATED; truth available" if data.has_truth else "RECORDING; independent truth unavailable")
if not data.has_truth:
    print("Recording labelled synthetic:", data.metadata.get("synthetic", False))
print("Samples:", len(data.time), "duration [s]:", data.time[-1] - data.time[0])


## E3: change assumptions, not the sensor noise

Predict the effect of multiplying **R by 0.1 and 10**, then inspect the results. Standard deviations scale with the square root of those numbers. Choose `candidate_acceleration_std` for a Q comparison, in m/s², and explain its effect. The actual data stay fixed.

`compare` handles saving and plotting through the same functions as the command-line runner. Open each case's `diagnostics.png` and the saved `summary.json` for per-sensor pulls, NIS and rejected counts. Simulation adds truth-based errors and coverage. Separate startup from later intervals.


In [ ]:
baseline = FilterConfig()
candidate_acceleration_std = 0.08  # Your Q assumption: predict before running.

cases = [ExperimentCase("baseline", baseline, "nominal")]
for multiplier in (0.1, 10.0):
    changed = replace(baseline,
                      position_noise_std=baseline.position_noise_std * np.sqrt(multiplier),
                      flow_velocity_noise_std=baseline.flow_velocity_noise_std * np.sqrt(multiplier),
                      range_noise_std=baseline.range_noise_std * np.sqrt(multiplier))
    cases.append(ExperimentCase(f"R-{multiplier:g}", changed, "nominal"))
cases.append(ExperimentCase("Q-candidate", replace(baseline, acceleration_noise_std=candidate_acceleration_std), "nominal"))

e3_output = next_attempt(output / "E3")
summaries = compare(data, cases, e3_output, seed=seed, notes="Add our prediction and interpretation.")
display(Image(filename=str(e3_output / "comparison.png")))
display(Image(filename=str(e3_output / "baseline/diagnostics.png")))
display(summaries[0]["diagnostics"][:3])


## E4: implement your innovation decision

Replace the `None` in `our_innovation_threshold` with your dimension-aware NIS threshold. The worksheet gives the chi-square rule; explain its assumptions and your false-rejection probability. The unfinished function deliberately accepts everything, so running the notebook before editing it shows identical cases.

Use the same data source as E3, with one declared position outlier added by the supplied scenario. Inspect diagnostics **before rejection**, including rejected observations. Save the notebook to retain your implementation; optionally copy it to `exercise_policies.py` for command-line studies.


In [ ]:
from scipy.stats import chi2

false_rejection_probability = 0.01

def our_innovation_threshold(dimension, false_rejection_probability):
    # TODO E4: use the measurement dimension and justify your threshold.
    return None

outlier_data = select_data(recording, metadata, seed=seed, scenario="outlier")
gate_cases = [ExperimentCase("ungated", baseline, "outlier"),
              ExperimentCase("our-gate", baseline, "outlier", participant_gate=True)]
clean_cases = [replace(case, scenario="nominal") for case in gate_cases]
e4_clean_output = next_attempt(output / "E4-clean")
compare(data, clean_cases, e4_clean_output, seed=seed,
        alpha=false_rejection_probability, innovation_policy=our_innovation_threshold)
display(Image(filename=str(e4_clean_output / "comparison.png")))
e4_output = next_attempt(output / "E4")
gate_summaries = compare(outlier_data, gate_cases, e4_output, seed=seed,
                         alpha=false_rejection_probability,
                         innovation_policy=our_innovation_threshold)
display(Image(filename=str(e4_output / "comparison.png")))
display(Image(filename=str(e4_output / "our-gate/diagnostics.png")))
print("Rejected updates:", sum(row["rejected"] for row in gate_summaries[1]["diagnostics"]))


## E5: missing measurements and a stationary decision

First compare the same observations with and without a three-second position gap at 7-10 s. The input must extend to at least 12 s and contain position updates before, during and after that interval. Short recordings, or recordings with that gap already missing, are rejected before any comparison is saved. Choose a checked 30-second recording, or explicitly set both input paths to `None` and rerun the data-selection cell for the labelled synthetic fallback. Which uncertainties grow, and what constrains each axis?

The second comparison uses **controlled simulated rest and motion**, including constant-velocity translation. This supplies independent motion labels for testing your decision; the live helper does not provide raw flow for this study. Implement `our_stationary_decision` using the supplied window fields (duration, gyro, acceleration magnitude/scatter, motor status and fresh flow speed). Quiet IMU data alone can also occur during translation. Until you replace `False`, both cases omit ZUPT. Explain the missed and false stationary decisions, not just whether drift looks smaller.


In [ ]:
dropout_cases = [ExperimentCase("complete", baseline, "nominal"),
                 ExperimentCase("position-gap", baseline, "dropout", dropout_s=3.0)]
e5_dropout_output = next_attempt(output / "E5-dropout")
compare(data, dropout_cases, e5_dropout_output, seed=seed)
display(Image(filename=str(e5_dropout_output / "comparison.png")))


In [ ]:
def our_stationary_decision(window):
    # TODO E5: justify thresholds and require an independent motion cue.
    return False

stationary_data = select_data(seed=seed, scenario="stationary")
stationary_cases = [ExperimentCase("without-zupt", baseline, "stationary"),
                    ExperimentCase("our-zupt", baseline, "stationary", participant_stationary=True)]
e5_stationary_output = next_attempt(output / "E5-stationary")
stationary_summaries = compare(stationary_data, stationary_cases, e5_stationary_output, seed=seed,
                               stationary_policy=our_stationary_decision)
display(Image(filename=str(e5_stationary_output / "comparison.png")))
display(Image(filename=str(e5_stationary_output / "our-zupt/stationarity.png")))
print("False accepted stationary updates:", stationary_summaries[1]["false_stationary_updates"])


## E6: compare two filters on one capture

Use the capture cell above for the motor-free session, or the equivalent [terminal helper](../../docs/sensor_recording_contract.md#live-e6-session-and-replay-fallback). Both save two live filter outputs plus the input recording. Reuse that capture here; another acquisition is unnecessary. The cell below replays your selected capture, or generates a clearly labelled complete synthetic fallback if no capture is selected.

Choose a second position-noise assumption, predict the difference, and compare the saved position/innovation plots. Replaying the same capture isolates the filter choice from changes in hand motion. This does not change onboard estimation or qualify a controller to fly.


In [ ]:
from day2.replay import save_results
from kalman.recording import load_recording, replay

if data is None:
    raise ValueError("Rerun the data-selection cell successfully before continuing.")

e6_output = next_attempt(output / "E6")
e6_output.mkdir()
if recording is None:
    prepared = e6_output / "simulated-input"
    prepared.mkdir()
    export_prepared(data, ExperimentCase("prepared", baseline, "nominal"), prepared, seed)
    replay_csv, replay_metadata = prepared / "prepared.csv", prepared / "recording.json"
else:
    replay_csv, replay_metadata = Path(recording), Path(metadata)

samples, recording_details = load_recording(replay_csv, replay_metadata)
comparison_position_std = 0.12  # metres; choose and explain.
for name, standard_deviation in (("baseline", baseline.position_noise_std),
                                 ("comparison", comparison_position_std)):
    replay_config = replace(baseline, position_noise_std=standard_deviation)
    save_results(replay(samples, replay_config), recording_details, replay_config, e6_output / name)
    display(Image(filename=str(e6_output / f"{name}.png")))


## E7: turn uncertainty into a race-day choice

Use this single-gate example to discuss a crossing margin. Change the estimated position or the position standard deviations. Which directions reduce clearance? What must you know about physical tracking and survey errors before choosing a speed?

The supplied speed policy is a fixed baseline; editing its uncertainty input changes clearance, not simulated motion or lap time. Follow [E7](../../docs/kalman_filter_practical.md#e7-turn-covariance-into-a-racing-decision) to justify a candidate policy.


In [ ]:
from race_common.hoop import Hoop
from race_common.strategy import RaceLimits, recommended_speed

gate = Hoop(np.array([1.0, 0.0, 0.7]), np.array([1.0, 0.0, 0.0]), radius_m=0.5)
estimated_position = np.array([1.0, 0.03, 0.7])
position_std = np.array([0.02, 0.04, 0.03])  # metres
speed, clearance = recommended_speed(gate, estimated_position, np.diag(position_std**2), RaceLimits())
print("Baseline recommendation [m/s]:", speed, "predicted clearance [m]:", clearance)


## Discuss and keep

Save this notebook, the E2 calibration decision and links to one useful comparison. In the same short team notes, explain one failed prediction or uncertainty and one race-day implication. The saved plots/CSVs/settings carry the detail. Core timings and checks remain those in the worksheet; larger seed scans and extra models are optional.
